# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [2]:
import heapq # Biblioteca estándar de Python. Maneja listas como "Priority Queues" (Colas de Prioridad).

def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    # LA HEURÍSTICA PROPIA ---

    def heuristic(node):
        h = 0
        objetivo = node.state.rods[2] 
        
        for disco in range(1, number_disks + 1):
            
            # Preguntamos SI este disco en la vara donde debería estar al final?"
            if disco not in objetivo:
                
                # Si el disco NO está en la vara 3, sumamos su valor a h
                # Si falta el disco 5, h aumenta en 5 (penalización alta).
                # Si falta el disco 1, h aumenta solo en 1 (penalización baja).
                # Prioriza acomodar los discos grandes 
                # porque son los que más ensucian el puntaje.
                h += disco 
        return h

    start_node = NodeHanoi(initial_state)
    
    frontier = []
    heapq.heappush(frontier, (heuristic(start_node), start_node))
    explored = set()
    visited_states = {initial_state: 0}
    nodes_explored = 0
    max_depth = 0
    solution = None 

    while frontier:
        f_cost, current_node = heapq.heappop(frontier)
        nodes_explored += 1
        if problem.goal_test(current_node.state):
            solution = current_node 
            break

        # Agregamos la foto actual a explorados para no repetirla.
        explored.add(current_node.state)

        for child in current_node.expand(problem):
            g_cost = child.path_cost
            if child.state not in explored:
                if child.state not in visited_states or g_cost < visited_states[child.state]:
                    visited_states[child.state] = g_cost
                    f_val = g_cost + heuristic(child)
                    heapq.heappush(frontier, (f_val, child))
                    
                    if child.depth > max_depth:
                        max_depth = child.depth

    #  RETORNO DE RESULTADOS ---
    metrics = {
        "solution_found": solution is not None,
        "nodes_explored": nodes_explored,
        "states_visited": len(visited_states),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        # solution.path_cost es la cantidad total de pasos del camino óptimo.
        "cost_total": float(solution.path_cost) if solution else 0.0,
    }

    return solution, metrics

Se prueba la función:

In [3]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 158
states_visited: 163
nodes_in_frontier: 11
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
